# 8. Calculating polygon area

Spatial measurements only make sense when we understand the units of the coordinate reference system.

The Toronto ward file uses **EPSG:32617**, a projected CRS measured in metres.

| Operation | Resulting units |
|---|---|
| distance | metres |
| area | square metres |
| area ÷ 1,000,000 | square kilometres |

This is why we can safely calculate ward area from this file.

In [ ]:
import geopandas as gpd

wards = gpd.read_file("../data/city-wards.gpkg")

print("CRS:", wards.crs)
print("Geometry types:", wards.geometry.geom_type.unique())

## What does `.area` return?

GeoPandas calculates polygon area using the units of the layer's CRS.

Conceptually:

| Ward | Area in m² | Area in km² |
|---|---:|---:|
| Ward A | 12,500,000 | 12.5 |
| Ward B | 7,800,000 | 7.8 |

```text
square metres ÷ 1,000,000 = square kilometres
```

In [ ]:
wards["area_m2"] = wards.geometry.area
wards["area_km2"] = wards["area_m2"] / 1_000_000

wards[
    ["WARD_SHORT_CODE", "WARD_NAME", "area_km2"]
].head()

## Inspect the result

Sorting is useful as a plausibility check. Larger outer wards should generally have larger land areas than compact central wards.

In [ ]:
wards[
    ["WARD_NAME", "area_km2"]
].sort_values(
    "area_km2",
    ascending=False
).head(10)

## Why area matters

Raw counts can be misleading when geographic units differ greatly in size.

| Ward | Libraries | Area | Libraries per 10 km² |
|---|---:|---:|---:|
| Ward A | 4 | 10 km² | 4.0 |
| Ward B | 4 | 40 km² | 1.0 |

The raw count is identical, but the spatial concentration is very different.

Later, we can combine ward area with library counts to create a normalized measure.